# Задание 04. Данные из PubChem через API

Получим свойства веществ из официальной базы PubChem через PUG REST API.

## Шаг 1. Одно вещество по имени

Запросим SMILES, формулу и массу аспирина.

In [ ]:
import requests

url = ("https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/aspirin"
       "/property/CanonicalSMILES,InChI,InChIKey,MolecularFormula,MolecularWeight/JSON")
resp = requests.get(url)
print("Статус:", resp.status_code)
data = resp.json()
props = data["PropertyTable"]["Properties"][0]
props


## Шаг 2. Несколько веществ

Соберём таблицу по списку веществ.

In [ ]:
import pandas as pd

compounds = ["aspirin", "ibuprofen", "paracetamol", "caffeine", "ethanol"]
rows = []
for name in compounds:
    url = (f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{name}"
           f"/property/MolecularFormula,MolecularWeight,CanonicalSMILES/JSON")
    try:
        r = requests.get(url)
        p = r.json()["PropertyTable"]["Properties"][0]
        rows.append({"вещество": name, **p})
    except Exception as e:
        print("Ошибка для", name, e)
df = pd.DataFrame(rows)
df


## Шаг 3. CID и InChIKey

CID — числовой идентификатор; InChIKey — «отпечаток» вещества для проверки совпадений.

In [ ]:
cid_url = "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/aspirin/cids/TXT"
print("CID аспирина:", requests.get(cid_url).text.strip())
ik_url = ("https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/2244"
          "/property/InChIKey/TXT")
print("InChIKey аспирина:", requests.get(ik_url).text.strip())


## Шаг 4. Проверка агента через PubChem

Попросите агента дать SMILES кофеина, затем проверьте его через PubChem. Совпадает ли канонический SMILES?

In [ ]:
agent_smiles = "Cn1c(=O)c2c(ncn2C)n(C)c1=O"  # впишите SMILES, который дал агент

url = ("https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/caffeine"
       "/property/CanonicalSMILES/JSON")
p = requests.get(url).json()["PropertyTable"]["Properties"][0]
print("Агент:  ", agent_smiles)
print("PubChem:", p["CanonicalSMILES"])
from rdkit import Chem
a = Chem.MolToSmiles(Chem.MolFromSmiles(agent_smiles))
b = Chem.MolToSmiles(Chem.MolFromSmiles(p["CanonicalSMILES"]))
print("Совпадают после канонизации?", a == b)


## Шаг 5. Выводы

Что нового вы узнали об API? Как проверить, что данные «из PubChem», а не придуманы?

**Выводы:**

- ...